# 05 · The SO-101 on the mesh

**What you will see**

- The SO-101 and a G1 **mounted** on a mesh: each becomes a device with a presence token, a record (its manifest) and rpcs.
- **`discover()`** with selectors over type, labels and id.
- **`invoke('act')`** through the device's own safety chain, and an out-of-range write **refused verbatim** at the caller.
- **`subscribe('state')`** over the mesh, with the measured rate.
- **`connect(id)`**: a `Robot` whose body is on the other peer, running a policy (with video).
- **`broadcast('estop')`**: every device latches; who acked. Then **resume authority**: who may release it.
- **`mesh.tool`**: the `use_mesh` agent tool, generated from the manifests present.
- A **second OS process** hosting another arm, discovered, then killed and seen leaving.

Transport: zenoh over explicit loopback endpoints (scouting off), a tenant unique to this run.

In [1]:
import _nb
_nb.drawing("d07_mesh")

## Two meshes on loopback

`me` is the controller (this could be another machine). `bench` is where the robots live. Both are zenoh peers in one tenant.

In [2]:
import json, os, socket, subprocess, sys, threading, time, uuid
import numpy as np
from PIL import Image, ImageDraw
from strands_robots import Robot, Simulation
from strands_robots.mesh import Mesh, RequestRefused, authority

PORT = 8750  # this notebook's loopback port
TENANT = f"nb05-{uuid.uuid4().hex[:8]}"
ME_URL, BENCH_URL = f"zenoh://?listen=tcp/127.0.0.1:{PORT}", f"zenoh://?connect=tcp/127.0.0.1:{PORT}"

me = Mesh(ME_URL, tenant=TENANT, operator="alice@laptop")
bench = Mesh(BENCH_URL, tenant=TENANT, operator="bench@lab")
print(me); print(bench)

Mesh('zenoh://?listen=tcp/127.0.0.1:8750', tenant='nb05-db5f0e6b', mounted=[])
Mesh('zenoh://?connect=tcp/127.0.0.1:8750', tenant='nb05-db5f0e6b', mounted=[])


## Mount the arm and the G1

In [3]:
arm_world = Simulation("mujoco", render_size=(480, 360))
arm = Robot("so101", sim=arm_world)
arm_world.camera("cam", position=(0.55, -0.45, 0.4), target=(0.0, 0.0, 0.15), size=(480, 360))
g1 = Robot("g1", sim=Simulation())

d_arm = bench.mount(arm, labels={"role": "arm", "bench": "left"}, state_hz=100)
d_g1 = bench.mount(g1, labels={"role": "humanoid", "bench": "left"})
print(d_arm); print(d_g1)

Device('so101', type='robot_arm', tenant='nb05-db5f0e6b', labels={'role': 'arm', 'bench': 'left'})
Device('unitree_g1', type='humanoid', tenant='nb05-db5f0e6b', labels={'role': 'humanoid', 'bench': 'left'})


## `discover()` with selectors

A selector is `device(*)` or `device(<key>:<value>,...)` over the labels plus `id`, `type` and `tenant`; a bare id means `device(id:<id>)`.

In [4]:
t0 = time.perf_counter()
me.discover("device(*)", min_count=2, timeout=5.0)
print(f"both devices visible to the controller within {(time.perf_counter() - t0) * 1000:.0f} ms")
rows = []
for sel in ("device(*)", "device(type:robot_arm)", "device(role:humanoid)", "device(bench:left,type:humanoid)", "so101", "device(bench:right)"):
    rows.append({"selector": f"`{sel}`", "found": ", ".join(d.id for d in me.discover(sel)) or "—"})
_nb.show(_nb.table(rows))
d = me.discover("so101")[0]
print("record:", d.id, d.device_type, d.tenant, d.labels, "| manifest controls:", [c["name"] for c in d.manifest["controls"]])

both devices visible to the controller within 0 ms


| selector | found |
|---|---|
| `device(*)` | so101, unitree_g1 |
| `device(type:robot_arm)` | so101 |
| `device(role:humanoid)` | unitree_g1 |
| `device(bench:left,type:humanoid)` | unitree_g1 |
| `so101` | so101 |
| `device(bench:right)` | — |

record: so101 robot_arm nb05-db5f0e6b {'role': 'arm', 'bench': 'left'} | manifest controls: ['shoulder_pan.pos', 'shoulder_lift.pos', 'elbow_flex.pos', 'wrist_flex.pos', 'wrist_roll.pos', 'gripper.pos']


## `invoke()`: motion is the device's call

`act` over the mesh runs `confirm_act` on the device: the same safety chain and operator gate as in `04`. A remote caller never confirms for the operator.

In [5]:
obs = me.invoke("so101", "act", action={"shoulder_pan": 0.3}, steps=50)
print("act accepted; the device now reads shoulder_pan =", round(arm.observe().joints["shoulder_pan"], 3))
try:
    me.invoke("so101", "act", action={"shoulder_pan": 99.0})
except RequestRefused as remote_err:
    remote_msg = str(remote_err)
try:
    arm.confirm_act({"shoulder_pan": 99.0})
except RequestRefused as local_err:
    local_msg = str(local_err)
print("over the mesh:", remote_msg)
print("in-process   :", local_msg)
print("verbatim:", remote_msg == local_msg)
try:
    me.invoke("so101", "teleport", x=1)
except LookupError as e:
    print("unknown rpc:", e)

act accepted; the device now reads shoulder_pan = 0.242
over the mesh: so101: shoulder_pan=99.0000 rad is outside its range [-1.9199, 1.9199]
in-process   : so101: shoulder_pan=99.0000 rad is outside its range [-1.9199, 1.9199]
verbatim: True
unknown rpc: so101 has no rpc 'teleport' over the mesh; known: ['act', 'describe', 'estop', 'hold', 'lockout', 'observe', 'resume', 'stop']


## `subscribe('state')` over the mesh

The payload is the one `arm.subscribe('state', ...)` gives in-process. The arm was mounted with `state_hz=100`; a local run moves it while we listen.

In [6]:
def wave(obs):
    t = obs.t
    return {"shoulder_pan": 0.8 * np.sin(1.5 * t), "shoulder_lift": -0.4 + 0.3 * np.sin(t),
            "elbow_flex": 0.6 + 0.4 * np.sin(1.2 * t), "wrist_flex": 0.5 * np.sin(2 * t), "gripper": 0.6}

got = []
unsubscribe = me.subscribe("so101", "state", got.append)
task = arm.start(wave, hz=50, seconds=1.5, realtime=True)
time.sleep(0.2); got.clear(); time.sleep(1.0)
unsubscribe(); task.halt(3.0)
seqs = [g["seq"] for g in got]
rate = (len(got) - 1) / (got[-1]["t"] - got[0]["t"])
_nb.show(_nb.table([{"seq": g["seq"], "t − t₀ (s)": f"{g['t'] - got[0]['t']:.3f}",
                     **{k: f"{g[k]:+.4f}" for k in ("shoulder_pan.pos", "elbow_flex.pos")}} for g in got[::20]]))
print(f"{len(got)} samples in 1 s over zenoh: {rate:.1f} Hz (device publishes at 100), "
      f"gaps in seq: {sum(b - a - 1 for a, b in zip(seqs, seqs[1:]))}")

| seq | t − t₀ (s) | shoulder_pan.pos | elbow_flex.pos |
|---|---|---|---|
| 23 | 0.000 | +0.2813 | +0.8400 |
| 43 | 0.200 | +0.5261 | +0.8214 |
| 63 | 0.402 | +0.6803 | +0.9072 |
| 83 | 0.601 | +0.7734 | +0.9613 |
| 103 | 0.802 | +0.7996 | +0.9974 |

100 samples in 1 s over zenoh: 99.8 Hz (device publishes at 100), gaps in seq: 0


## `connect(id)`: a remote Robot, running a policy

`me.connect('so101')` returns a `Robot` whose body is the device on the bench. `run()` works unchanged; each tick is an `observe` and an `act` rpc, and every write passes the device's chain. The video is rendered on the bench's world between ticks.

In [7]:
arm.act({k: 0.0 for k in arm.action_keys}, steps=50)
remote = me.connect("so101")
print(remote.id, type(remote.body).__name__, "| position hold:", remote.supports_position_hold)

frames = []
def caption(img, text):
    im = Image.fromarray(img); d = ImageDraw.Draw(im)
    d.rectangle([0, 0, im.width, 22], fill=(20, 20, 20)); d.text((8, 5), text, fill=(240, 240, 240))
    return np.asarray(im)

def remote_policy(obs):
    if len(frames) < 400 and round(obs.t * 50) % 3 == 0:
        frames.append(caption(arm_world.render("cam"), f"me.connect('so101').run(policy)   t={obs.t:4.2f}s"))
    return wave(obs)

res = remote.run(remote_policy, hz=50, steps=200)
print(f"steps={res.steps} stopped_by={res.stopped_by!r} world time {res.seconds:.2f} s, {res.hz_achieved:.0f} ticks/s over the mesh (not paced: realtime=False)")
print("remote reads == device reads:", np.allclose(remote.observe().joints.positions, arm.observe().joints.positions, atol=1e-9))
_nb.video(frames, fps=15)

so101 RemoteBody | position hold: True


steps=200 stopped_by='steps' world time 4.00 s, 50 ticks/s over the mesh (not paced: realtime=False)
remote reads == device reads: True


## `broadcast('estop')`: one stop for the tenant

It goes to the tenant's `@estop` key (every mesh latches its matching devices on hearing it) and to each match as an rpc whose replies are the acks. Never retained, never queued. Here it lands while the arm is running.

In [8]:
task = arm.start(wave, hz=50, seconds=30, realtime=True)
time.sleep(0.5)
t0 = time.perf_counter()
acked = me.broadcast("estop", selector="device(*)")
dt = (time.perf_counter() - t0) * 1000
task.halt(2.0)
print(f"acked by {acked} in {dt:.1f} ms")
print("arm run:", task.result.stopped_by, "|", task.result.fault)
_nb.table([{"device": r.id, "estopped": r.estopped, "lockout": me.invoke(r.id, "lockout")} for r in (arm, g1)])

acked by ['so101', 'unitree_g1'] in 1.4 ms
arm run: estop | None


| device | estopped | lockout |
|---|---|---|
| so101 | True | {'engaged': True, 'epoch': '0868e970257d495d98823f86868cfee4', 'by': 'alice@laptop'} |
| unitree_g1 | True | {'engaged': True, 'epoch': '0868e970257d495d98823f86868cfee4', 'by': 'alice@laptop'} |

## Who may resume

The operator who stopped it, or a name in `Authority(names=...)`, and names are only trusted on an in-process or loopback transport (as here). On any other network the device wants a fresh Ed25519-signed assertion for that stop and that device.

In [9]:
for label, call in [("mallory, invoke", lambda: me.invoke("so101", "resume", approved_by="mallory")),]:
    try:
        call()
    except RequestRefused as e:
        print(f"{label}: refused → {e}")
print("mallory, broadcast → acked", me.broadcast("resume", selector="device(*)", approved_by="mallory"), "| still latched:", arm.estopped, g1.estopped)

bench.resume_authority = authority.Authority(names=("safety-lead",))
print("safety-lead resumes the G1 → acked", me.broadcast("resume", selector="unitree_g1", approved_by="safety-lead"), "| g1 latched:", g1.estopped)
me.invoke("so101", "resume", approved_by=me.operator)
print(f"{me.operator} (who stopped it) resumes the arm → latched: {arm.estopped}")

mallory, invoke: refused → so101: 'mallory' is not a resume authority for this stop (allowed: ['alice@laptop']). Resume as the operator who stopped it (broadcast/estop by=<operator>), name one in Mesh(resume_authority=Authority(names=[...])), or use operator keys (Authority(public_keys=...))
mallory, broadcast → acked [] | still latched: True True
safety-lead resumes the G1 → acked ['unitree_g1'] | g1 latched: False
alice@laptop (who stopped it) resumes the arm → latched: False


In [10]:
# Off loopback: only signed resumes. Same devices, keys instead of names.
private, public = authority.keygen()
bench.resume_authority = authority.Authority(public_keys=(public,))
me.broadcast("estop", selector="so101")
try:
    me.invoke("so101", "resume", approved_by=me.operator)
except RequestRefused as e:
    print("unsigned:", e)
me.resume_key = private
me.invoke("so101", "resume", approved_by=me.operator)
print("signed for this stop and this device → latched:", arm.estopped, "| lockout:", me.invoke("so101", "lockout"))
me.resume_key = None
bench.resume_authority = authority.Authority()

unsigned: so101: resume needs an assertion signed by an operator key
signed for this stop and this device → latched: False | lockout: {'engaged': False, 'epoch': None, 'by': None}


## `mesh.tool`: `use_mesh` for an agent

One strands tool generated from the manifests of the devices present. A refusal goes back to the model as text; the model can stop but never resume.

In [11]:
tool = me.tool
spec = tool.tool_spec
print(tool.tool_name); print(spec["description"]); print()
schema = spec["inputSchema"]["json"]
print("actions:", schema["properties"]["action"]["enum"])
print("act keys:", sorted(schema["properties"]["act"]["properties"]))
for k, v in schema["properties"].items():
    print(f"  {k:12s} {v.get('type', '')}  {(v.get('description') or '')[:90]}")
print("required:", schema.get("required"))

use_mesh
Every device on the mesh (tenant nb05-db5f0e6b). Selectors: device(*), device(role:arm,...).
- so101 (robot_arm; bench:left,role:arm) rpcs=['describe', 'observe', 'act', 'stop', 'hold', 'estop', 'resume']
- unitree_g1 (humanoid; bench:left,role:humanoid) rpcs=['describe', 'observe', 'act', 'stop', 'hold', 'estop', 'resume']

actions: ['list', 'describe', 'observe', 'act', 'run', 'stop', 'estop', 'resume']
act keys: ['elbow_flex', 'gripper', 'left_ankle_pitch_joint', 'left_ankle_roll_joint', 'left_elbow_joint', 'left_hip_pitch_joint', 'left_hip_roll_joint', 'left_hip_yaw_joint', 'left_knee_joint', 'left_shoulder_pitch_joint', 'left_shoulder_roll_joint', 'left_shoulder_yaw_joint', 'left_wrist_pitch_joint', 'left_wrist_roll_joint', 'left_wrist_yaw_joint', 'right_ankle_pitch_joint', 'right_ankle_roll_joint', 'right_elbow_joint', 'right_hip_pitch_joint', 'right_hip_roll_joint', 'right_hip_yaw_joint', 'right_knee_joint', 'right_shoulder_pitch_joint', 'right_shoulder_roll_joint', 'ri

In [12]:
def call(**inp):
    r = tool._tool_func({"toolUseId": "nb", "input": inp})
    return r["status"], r["content"][0]["text"][:200]

for inp in ({"action": "list", "selector": "device(role:arm)"},
            {"action": "act", "device_id": "so101", "act": {"shoulder_pan": 99.0}},
            {"action": "estop", "selector": "so101"},
            {"action": "resume", "device_id": "so101", "approved_by": me.operator}):
    print(inp["action"], "→", *call(**inp))
me.invoke("so101", "resume", approved_by=me.operator)   # the operator's own code does
print("operator resumed in code → latched:", arm.estopped)

list → success [{"id": "so101", "device_type": "robot_arm", "labels": {"role": "arm", "bench": "left"}, "last_seen": 1792036736.250158}]
act → error RequestRefused: so101: shoulder_pan=99.0000 rad is outside its range [-1.9199, 1.9199]
estop → success ["so101"]
resume → error PermissionError: resume is the operator's, not an agent's: the operator clears so101 with mesh.invoke(id, 'resume', approved_by=...) or the dashboard
operator resumed in code → latched: False


## A device in another OS process

Same python, same package, one more SO-101 in its own process and world, joining the controller's loopback endpoint.

In [13]:
peer_code = '''
import sys, time
from strands_robots import Robot
from strands_robots.mesh import Mesh
url, tenant = sys.argv[1:3]
mesh = Mesh(url, tenant=tenant, operator="bench-b@lab")
mesh.mount(Robot("so101", id="so101-b"), labels={"role": "arm", "bench": "right"})
print("ready", flush=True)
while True:
    time.sleep(1)
'''
env = {**os.environ, "PYTHONPATH": os.pathsep.join([str(_nb.ROOT), os.getcwd()])}
peer = subprocess.Popen([sys.executable, "-c", peer_code, BENCH_URL, TENANT], stdout=subprocess.PIPE, text=True, env=env)
guard = threading.Timer(120, peer.kill); guard.start()
assert peer.stdout.readline().strip() == "ready"
found = me.discover("device(bench:right)", min_count=1, timeout=5.0)
print("pid", peer.pid, "→", found)
print("all arms:", [d.id for d in me.discover("device(role:arm)")])
print("its describe():", {k: me.invoke("so101-b", "describe")[k] for k in ("device_id", "device_type", "real", "world")})
print("estop to device(role:arm) acked by", me.broadcast("estop", selector="device(role:arm)"))
me.invoke("so101", "resume", approved_by=me.operator)

pid 77785 → [Device('so101-b', type='robot_arm', tenant='nb05-db5f0e6b', labels={'role': 'arm', 'bench': 'right'})]
all arms: ['so101', 'so101-b']
its describe(): {'device_id': 'so101-b', 'device_type': 'robot_arm', 'real': False, 'world': 'mujoco'}
estop to device(role:arm) acked by ['so101', 'so101-b']


In [14]:
peer.kill()
t0 = time.perf_counter()
while any(d.id == "so101-b" for d in me.discover()) and time.perf_counter() - t0 < 10:
    time.sleep(0.01)
print(f"SIGKILL → left discover() after {(time.perf_counter() - t0) * 1000:.0f} ms (presence ttl {me.presence_ttl:g} s)")
print("present now:", [d.id for d in me.discover()])
peer.wait(5); guard.cancel()
print("peer exit code:", peer.returncode)

SIGKILL → left discover() after 24 ms (presence ttl 3 s)
present now: ['so101', 'unitree_g1']
peer exit code: -9


## The same mesh over AWS IoT Core

`Mesh("iot://us-west-2")` runs these calls as confirmed Direct Messages to the one client that holds the device. Not run here; measured on 2026-10-15 against a real account with a simulated SO-101:

> real AWS discover 0.83 s, act p50 203 / p99 604 ms, estop acked, estop to a killed device → 404 (nothing queued); the thing, policy and certificate were deleted and verified gone.

On loopback zenoh the same build measured invoke(act) p50 1.17 ms / p99 4.2 ms and state at 100 / 200 / 500 Hz with no drops.

In [15]:
me.close(); bench.close()
arm_world.close(); g1.sim.close()
if peer.poll() is None:
    peer.kill()
print("meshes closed, worlds closed, peer process gone:", peer.poll() is not None)

meshes closed, worlds closed, peer process gone: True


## Where this sits in v1

One `Mesh` per process, many transports, one id space. It reads only what `04` showed, the device's manifest, and every write it carries lands in the device's own safety chain, so a robot on the mesh is exactly as safe as the robot in your hand. `connect()` makes a remote body look like any other, which is how a policy server, a dashboard or an agent on another machine drives the arm. The physical SO-101 behind the operator gate, and an estop measured to its last servo write, are the rows still waiting for the bench.